# 05 — Prediction drift and delayed labels

When ground truth arrives days later, **score distributions** are legitimate early warnings. This notebook:

1. Compares predicted probabilities between an early and a late window on a covariate stream (frozen model).
2. Simulates **label latency** and shows how rolling outcome metrics appear only after joins catch up.


In [ ]:
# From repo root: pip install -e ".[dev]"
%matplotlib inline

import pandas as pd

from drift_lab import StreamConfig, build_ledger_route_model, generate_stream
from drift_lab.analysis import attach_predictions, simulate_label_delay
from drift_lab.metrics import rolling_accuracy, two_sample_ks
from drift_lab.viz import prediction_hist_figure, rolling_accuracy_figure

cfg = StreamConfig()
model = build_ledger_route_model(cfg)
df = generate_stream("covariate_gradual", cfg)
scored = attach_predictions(model, df)

early = scored[(scored["day"] >= 30) & (scored["day"] < 50)]
late = scored[scored["day"] >= 90]
ks_stat, ks_p = two_sample_ks(early["prob_review"], late["prob_review"])
print(f"KS on scores: stat={ks_stat:.4f}, p={ks_p:.2e}")


## Prediction distribution shift


In [ ]:
fig, hist_meta = prediction_hist_figure(
    early["prob_review"].values,
    late["prob_review"].values,
)
hist_meta


## Delayed label join (synthetic 4-day latency)


In [ ]:
delayed = simulate_label_delay(scored, latency_days=4)
# Labels for transactions on day d become available on day d+4
joinable = delayed[delayed["label_available_day"] <= delayed["day"].max()]
print(f"Joinable rows: {len(joinable):,} / {len(delayed):,}")

acc_df = rolling_accuracy(
    joinable["label"].values,
    joinable["pred_label"].values,
    window=800,
)
fig, roll_meta = rolling_accuracy_figure(
    acc_df,
    title="Rolling accuracy after label join (4-day latency)",
)
roll_meta


## Outcome accuracy: early vs late window


In [ ]:
def window_acc(part):
    return (part["pred_label"] == part["label"]).mean()

print("Early days 30–49:", round(window_acc(early), 4))
print("Late days 90+:", round(window_acc(late), 4))


## Takeaway

- Log **`expense_id`, model_version, feature_schema, score, timestamp** on every prediction.
- Use score drift for **triage**, not automatic retrain, when labels lag.
- Pair quantile shift with a small **audit batch** or shadow threshold before paging.
